# 27_01 RUL 기반 고장 임박 문제

In [ ]:
# [환경 설정] 한글 폰트 설정 및 필수 라이브러리 로드
# macOS, Windows, Linux, Google Colab 환경에 맞춰 한글 깨짐 없이 동작하도록 자동 감지 설정합니다.

import platform
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import pandas as pd
import numpy as np

try:
    import seaborn as sns
except ImportError:
    pass

# 운영체제(OS)별 한글 폰트 자동 설정
system_name = platform.system()
if system_name == 'Darwin':          # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
    plt.rcParams['font.sans-serif'] = ['AppleGothic', 'Apple SD Gothic Neo', 'NanumGothic', 'DejaVu Sans']
elif system_name == 'Windows':       # Windows
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['font.sans-serif'] = ['Malgun Gothic', 'NanumGothic', 'DejaVu Sans']
else:                               # Linux / Google Colab
    try:
        nanum_fonts = [f.name for f in fm.fontManager.ttflist if 'Nanum' in f.name]
        if nanum_fonts:
            plt.rcParams['font.family'] = nanum_fonts[0]
        else:
            import subprocess
            subprocess.run(['apt-get', 'install', '-y', 'fonts-nanum'], check=False, stdout=subprocess.DEVNULL)
            fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
            plt.rcParams['font.family'] = 'NanumGothic'
    except Exception:
        pass
    plt.rcParams['font.sans-serif'] = ['NanumGothic', 'DejaVu Sans']

# 마이너스 기호 깨짐 방지 및 Seaborn 폰트 동기화
plt.rcParams['axes.unicode_minus'] = False
try:
    if 'sns' in locals():
        sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])
except Exception:
    pass

print(f'✅ 환경 설정 완료! 현재 적용된 폰트: {plt.rcParams["font.family"]}')

## 라이브러리 임포트와 첫 불러오기
- 내내 다룰 데이터를 컴퓨터로 불러와 첫인사를 나누는 단계
- pandas를 불러오고 cmapss_fd001_sample.csv를 표로 읽어 첫 모습 확인

### 불러오기 코드
pandas를 pd로 부르고 read_csv로 파일을 df라는 표에 담기

In [ ]:
# 코드

### 첫 모습 확인
head로 첫 5행을, shape로 행과 열 개수를 확인

In [ ]:
# 코드

### 데이터 크기 점검
shape로 전체 행과 열의 개수를 다시 확인해 규모 감을 잡기

In [ ]:
# 코드

### 컬럼 이름 확인
분석 전 데이터가 어떻게 생겼는지 살펴보는 일은 길 떠나기 전 지도 확인

In [ ]:
# 코드

### 자료형과 결측 요약
info는 각 컬럼의 자료형과 값 개수를 요약 — 결측 단서까지 한눈에

In [ ]:
# 코드

### 기초 통계 살펴보기
describe는 숫자 컬럼의 평균·최소·최대 같은 기초 통계 한눈에

In [ ]:
# 코드

### 마지막 행 확인
tail로 마지막 5행을 보고 데이터가 끝까지 잘 들어왔는지 확인

In [ ]:
# 코드

### 설비 대수 세기
nunique는 서로 다른 값의 개수 — unit_id에 적용해 설비 대수 확인

In [ ]:
# 코드

## 결측치와 센서 범위 점검
- 빠진 값과 센서마다 값 크기 차이 — 분석 전 빈틈 점검부터
- 컬럼별 결측치 개수를 세고 센서값 범위를 비교
- 좋은 분석은 데이터의 빈틈을 먼저 점검하는 데서 시작


### 결측치 세기
isna와 sum을 이어 쓰면 컬럼별 결측치 개수 한눈에

In [ ]:
# 코드

### 한 센서 분포 보기
특정 센서 하나에 describe를 적용해 값의 분포를 자세히 보기

In [ ]:
# 코드

## 한 설비의 일생 따라가기
- 1번 엔진 하나만 떼어 내어 처음부터 고장까지 행을 따라 읽기
- 설비 한 대만 골라 cycle 순서로 RUL과 센서 변화 관찰
- 전체가 아니라 1번 엔진 하나에 집중

### 여러 센서 평균 비교
센서 여러 개의 평균을 한 번에 비교

In [ ]:
# 코드

### 한 설비만 골라내기
unit_id가 1인 행만 골라 unit1에 담고 행 수 확인

In [ ]:
# 코드

### 시작과 끝의 RUL 비교
이 설비의 시작 부분과 끝부분 RUL을 한눈에 비교

In [ ]:
# 코드

### RUL 변화 그래프 그리기
cycle에 따른 RUL 변화를 선 그래프로 그려 우하향 직선 확인

In [ ]:
# 코드

## 두 설비의 수명 비교
- 설비끼리는 cycle만으로 비교 불가 — 두 설비를 나란히 두고 직접 확인
- 두 설비의 최대 cycle 비교 + 같은 cycle에서 상태 차이
- 설비마다 고장까지 버티는 cycle 수가 다름을 직접 확인

### 설비별 수명 길이 구하기
groupby로 설비별 최대 cycle(=수명 길이)을 구하고 정렬


In [ ]:
# 코드

### 같은 cycle에서 RUL 비교
두 설비의 같은 50 cycle 시점을 각각 골라 RUL 비교

## 비교에서 얻는 교훈
- 같은 cycle이라도 설비마다 RUL이 다름
- cycle만으로는 설비끼리 직접 비교할 수 없음